# Lab 1: Intro to TensorFlow/Keras and Music Generation with RNNs

In this lab you'll get exposure to TensorFlow and Keras, and learn how they can
be used for deep learning.  Go through the code and run each cell.  Along the
way you'll encounter several ***TODO*** blocks — follow the instructions to fill
them out before running those cells and continuing.

# Part 1: Intro to TensorFlow / Keras

## 0.1 Install TensorFlow

[TensorFlow](https://www.tensorflow.org/) is Google's open-source machine
learning framework.  [Keras](https://keras.io/) ships as the high-level API
bundled inside TensorFlow (`tf.keras`).  For this lab we use TensorFlow 2.x
with the eager execution default.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)


## 1.1 What is TensorFlow?

TensorFlow is a machine learning library.  At its core it provides an interface
for creating and manipulating **tensors** — multi-dimensional arrays of base
datatypes such as integers or floats.  `tf.Tensor` objects are immutable
constant values (use `tf.Variable` for mutable state).

The [`shape`](https://www.tensorflow.org/api_docs/python/tf/TensorShape) of a
tensor defines its number of dimensions and the size of each dimension.
`ndim` (or `rank`) gives the number of dimensions.

Let's create some tensors and inspect their properties:


In [ ]:
integer = tf.constant(1234)
decimal = tf.constant(3.14159265359)

print(f"`integer` is a {integer.ndim}-d Tensor: {integer}")
print(f"`decimal` is a {decimal.ndim}-d Tensor: {decimal}")


Vectors and lists can be used to create 1-d tensors:

In [ ]:
fibonacci = tf.constant([1, 1, 2, 3, 5, 8])
count_to_100 = tf.constant(list(range(100)))

print(f"`fibonacci` is a {fibonacci.ndim}-d Tensor with shape: {fibonacci.shape}")
print(f"`count_to_100` is a {count_to_100.ndim}-d Tensor with shape: {count_to_100.shape}")


Next, let's create 2-d (matrices) and higher-rank tensors.  In image processing
we use 4-d Tensors with dimensions `(batch, height, width, channels)` — note
that TensorFlow defaults to **NHWC** order (channels last), the opposite of
PyTorch's NCHW.


In [ ]:
### Defining higher-order Tensors ###

'''TODO: Define a 2-d Tensor'''
matrix = # TODO

assert isinstance(matrix, tf.Tensor), "matrix must be a tf.Tensor"
assert matrix.ndim == 2

'''TODO: Define a 4-d Tensor.'''
# Use tf.zeros to initialize a 4-d Tensor of zeros with size (10, 256, 256, 3).
#   That is: 10 images, each 256x256 pixels, 3 RGB channels (NHWC format).
images = # TODO

assert isinstance(images, tf.Tensor), "images must be a tf.Tensor"
assert images.ndim == 4, "images must have 4 dimensions"
assert images.shape == (10, 256, 256, 3), "images is incorrect shape (expected NHWC)"
print(f"images is a {images.ndim}-d Tensor with shape: {images.shape}")


As you have seen, the `shape` of a tensor provides the number of elements in
each dimension.  You can also use slicing to access sub-tensors within a
higher-rank tensor:


In [ ]:
row_vector    = matrix[1]
column_vector = matrix[:, 1]
scalar        = matrix[0, 1]

print(f"`row_vector`: {row_vector}")
print(f"`column_vector`: {column_vector}")
print(f"`scalar`: {scalar}")


## 1.2 Computations on Tensors

A convenient way to think about and visualise computations in TensorFlow is in
terms of a **computation graph**.  We can define the graph in terms of tensors
(data) and the operations that act on them.  Let's look at a simple example:

![add graph](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/add-graph.png)


In [ ]:
# Create the nodes in the graph and initialise values
a = tf.constant(15)
b = tf.constant(61)

# Add them!
c1 = tf.add(a, b)
c2 = a + b   # TF overrides + to act on tensors

print(f"c1: {c1}")
print(f"c2: {c2}")


Notice how the output is a tensor with value 76.

Now let's consider a slightly more complicated example:

![computation graph](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/computation-graph.png)

We take two inputs `a, b` and compute an output `e`.  Each node represents an
operation.  Let's define a simple function in TensorFlow to construct this
computation:


In [ ]:
### Defining Tensor computations ###

def func(a, b):
    '''TODO: Define the operations for c, d, e.'''
    c = # TODO
    d = # TODO
    e = # TODO
    return e


Now we can call the function to execute the computation graph:

In [ ]:
a, b = 1.5, 2.5
e_out = func(a, b)
print(f"e_out: {e_out}")


Notice how our output is a tensor whose value is defined by the computation,
with no shape — it is a single scalar value.


## 1.3 Neural networks in Keras

We define neural networks in Keras.  The base building block is
[`tf.keras.layers.Layer`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Layer),
which is the Keras equivalent of PyTorch's `nn.Module`.

Consider a simple perceptron: $y = \sigma(Wx + b)$, where $W$ is a weight
matrix, $b$ a bias, $x$ the input, $\sigma$ the sigmoid activation, and $y$
the output.

![dense layer](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/computation-graph-2.png)

We subclass `tf.keras.layers.Layer` and override **`call()`** (the Keras
equivalent of PyTorch's `forward()`).  Weights are created with
`self.add_weight()` inside `build()` or `__init__`.


In [ ]:
### Defining a dense layer ###

class OurDenseLayer(tf.keras.layers.Layer):
    def __init__(self, num_outputs):
        super(OurDenseLayer, self).__init__()
        self.num_outputs = num_outputs

    def build(self, input_shape):
        # Initialise W and b as trainable weights.
        # Note: weight initialisation is random by default.
        self.W = self.add_weight(
            shape=(int(input_shape[-1]), self.num_outputs),
            initializer='random_normal', trainable=True
        )
        self.bias = self.add_weight(
            shape=(self.num_outputs,),
            initializer='random_normal', trainable=True
        )

    def call(self, x):
        '''TODO: define the operation for z (hint: use tf.matmul).'''
        z = # TODO

        '''TODO: define the operation for y (hint: use tf.sigmoid).'''
        y = # TODO
        return y


Now, let's test the output of our layer.

In [ ]:
# Define a layer and test the output!
num_inputs  = 2
num_outputs = 3
layer   = OurDenseLayer(num_outputs)
x_input = tf.constant([[1, 2.]])
y = layer(x_input)

print(f"input shape:  {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y}")


Conveniently, Keras provides built-in layers such as
[`tf.keras.layers.Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)
(the equivalent of PyTorch's `nn.Linear`) and activation layers.

Now, instead of a single custom layer, we'll use the
[`tf.keras.Sequential`](https://www.tensorflow.org/api_docs/python/tf/keras/Sequential)
API with a single `Dense` layer to define our network.


In [ ]:
### Defining a neural network using the Keras Sequential API ###

n_input_nodes  = 2
n_output_nodes = 3

# Define the model.
# Note: Keras Dense layers combine the linear transform and activation,
#       or you can stack layers explicitly.
'''TODO: Use Sequential to define a model with a single Dense layer
   followed by a sigmoid activation.'''
model = tf.keras.Sequential([ '''TODO''' ])


We've defined our model using the Sequential API.  Now let's test it:

In [ ]:
x_input      = tf.constant([[1, 2.]])
model_output = model(x_input)

print(f"input shape:  {x_input.shape}")
print(f"output shape: {model_output.shape}")
print(f"output result: {model_output}")


With Keras, we can create more flexible models by subclassing
[`tf.keras.Model`](https://www.tensorflow.org/api_docs/python/tf/keras/Model)
(the high-level equivalent of `tf.keras.layers.Layer` for full models).

Let's define the same architecture — a linear layer followed by a sigmoid — now
using subclassing and Keras's built-in `Dense` and `Activation` layers.


In [ ]:
### Defining a model using subclassing ###

class LinearWithSigmoidActivation(tf.keras.Model):
    def __init__(self, num_outputs):
        super(LinearWithSigmoidActivation, self).__init__()
        '''TODO: define a model with a single Dense layer and sigmoid activation.'''
        self.linear     = '''TODO: Dense layer'''
        self.activation = '''TODO: Activation layer'''

    def call(self, inputs):
        linear_output = self.linear(inputs)
        output        = self.activation(linear_output)
        return output


Let's test the model using an example input with `n_input_nodes=2` and
`n_output_nodes=3` as before.


In [ ]:
n_input_nodes  = 2
n_output_nodes = 3
model   = LinearWithSigmoidActivation(n_output_nodes)
x_input = tf.constant([[1, 2.]])
y = model(x_input)

print(f"input shape:  {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y}")


`tf.keras.Model` affords a lot of flexibility to define custom models.  For
example, we can use boolean arguments in `call()` to specify different network
behaviours — for instance, returning the input unchanged (an identity
operation).  Let's define a boolean `isidentity` argument to control this:


In [ ]:
### Custom behavior with subclassing tf.keras.Model ###

class LinearButSometimesIdentity(tf.keras.Model):
    def __init__(self, num_outputs):
        super(LinearButSometimesIdentity, self).__init__()
        self.linear = layers.Dense(num_outputs)

    '''TODO: Implement the behavior where the network outputs the input unchanged
       when isidentity=True.'''
    def call(self, inputs, isidentity=False):
        '''TODO'''


Let's test this behavior:

In [ ]:
# Test the IdentityModel
model   = LinearButSometimesIdentity(num_outputs=3)
x_input = tf.constant([[1, 2.]])

'''TODO: call the model with and without the identity option.'''
out_with_linear   = # TODO
out_with_identity = # TODO

print(f"input: {x_input}")
print("Network linear output: {}; network identity output: {}".format(
    out_with_linear, out_with_identity))


Now that we have learned how to define layers and models in Keras using both the
Sequential API and subclassing `tf.keras.Model`, we're ready to turn our
attention to how to actually implement network training with backpropagation.


## 1.4 Automatic Differentiation in TensorFlow

In TensorFlow, automatic differentiation is performed using
[`tf.GradientTape`](https://www.tensorflow.org/api_docs/python/tf/GradientTape).
Operations executed inside the `with tf.GradientTape() as tape:` context are
recorded; afterwards we call `tape.gradient(target, sources)` to compute
derivatives — the Keras/TF equivalent of PyTorch's `.backward()`.

Variables to differentiate with respect to must either be `tf.Variable` objects
(watched automatically) or tensors explicitly watched with `tape.watch()`.

Let's compute the gradient of $y = x^2$:


In [ ]:
### Gradient computation ###

# y = x^2
# Example: x = 3.0
x = tf.Variable(3.0)

with tf.GradientTape() as tape:
    y = x ** 2

dy_dx = tape.gradient(y, x)
print("dy_dx of y=x^2 at x=3.0 is:", dy_dx.numpy())
assert dy_dx.numpy() == 6.0


In training neural networks we use differentiation and stochastic gradient
descent (SGD) to optimise a loss function.  Let's find the minimum of
$L = (x - x_f)^2$ using `tf.GradientTape` and gradient descent.  While the
analytic solution is $x_{\min} = x_f$, working through this with TF's autograd
sets us up nicely for future labs.


In [ ]:
### Function minimization with GradientTape and gradient descent ###

x = tf.Variable(tf.random.normal([1]))
print(f"Initializing x={x.numpy()[0]:.4f}")

learning_rate = 1e-2
history = []
x_f = 4.0   # target value

for i in range(500):
    with tf.GradientTape() as tape:
        # TODO: compute the loss as the square of the difference between x and x_f
        loss = # TODO

    # Compute gradient and apply update
    grad = tape.gradient(loss, x)
    x.assign_sub(learning_rate * grad)
    history.append(x.numpy()[0])

# Plot the evolution of x as we optimise toward x_f!
plt.plot(history)
plt.plot([0, 500], [x_f, x_f])
plt.legend(('Predicted', 'True'))
plt.xlabel('Iteration')
plt.ylabel('x value')
plt.show()


We have now covered the fundamental concepts of TensorFlow/Keras — tensors,
operations, neural networks, and automatic differentiation.  Fire!!


---

## Part 2 Extension: Music Generation with HuggingFace

Instead of training an RNN from scratch, this section lets you pick a
**pre-trained model from HuggingFace Hub** and generate music immediately.

Two models are available, selectable in the next cell:

| Model | Approach | Input | Output |
|---|---|---|---|
| `facebook/musicgen-small` | Transformer seq2seq | **text prompt** | raw audio (WAV) |
| `sander-wood/tunesformer` | Causal LM | **ABC notation seed** | ABC notation text |

> **ABC notation** is a text-based music format used for Irish/Celtic folk tunes.
> Example: `X:1\nT:Title\nM:6/8\nK:Gmaj\n|: G2A B2c | d2e fed |`
> The model extends that seed, producing a complete tune you can play with `music21` or `abc2midi`.


In [ ]:
## ── Install HuggingFace dependencies ────────────────────────────────────────
## Run once; restart the kernel after installing if needed.

!pip install transformers accelerate scipy soundfile music21 --quiet


In [ ]:
## ── Model selector ──────────────────────────────────────────────────────────
## Change MUSIC_MODEL to switch between the two approaches at runtime.

# ── Pick one ─────────────────────────────────────────────────────────────────
MUSIC_MODEL = "musicgen"     # "musicgen"  |  "tunesformer"

# ── MusicGen config (used when MUSIC_MODEL == "musicgen") ────────────────────
#   Model sizes: musicgen-small (~300 MB) | musicgen-medium (~1.5 GB) | musicgen-large (~3.3 GB)
MUSICGEN_REPO      = "facebook/musicgen-small"
MUSICGEN_PROMPT    = "upbeat Irish folk music with fiddle and flute, lively jig"
MUSICGEN_DURATION  = 8        # seconds of audio to generate

# ── TunesFormer config (used when MUSIC_MODEL == "tunesformer") ──────────────
#   TunesFormer generates ABC notation conditioned on a control-code seed.
#   Seed format:  X:<index>  T:<title>  M:<time sig>  K:<key>  then barlines.
TUNESFORMER_REPO        = "sander-wood/tunesformer"
TUNESFORMER_SEED        = "X:1\nT:My Generated Tune\nM:6/8\nL:1/8\nK:Gmaj\n|: G2A B2c |"
TUNESFORMER_NEW_TOKENS  = 400   # max new tokens (≈ 1-2 full tunes)
TUNESFORMER_TEMPERATURE = 0.9
TUNESFORMER_TOP_K       = 50

OUTPUT_WAV = "generated_music.wav"

print(f"Selected model: {MUSIC_MODEL!r}")


### Option A — `facebook/musicgen-small` (text-prompt → audio)

MusicGen is a Transformer encoder-decoder trained by Meta AI on 20 000 hours of
licensed music.  You describe the music you want in plain English and it generates
a raw audio waveform directly — no ABC notation, no MIDI intermediate step.

Run the cell below when `MUSIC_MODEL = "musicgen"`.


In [ ]:
if MUSIC_MODEL == "musicgen":
    from transformers import pipeline
    import scipy.io.wavfile
    import numpy as np
    import IPython.display as ipd

    print(f"Loading {MUSICGEN_REPO} ...")
    musicgen_pipe = pipeline(
        "text-to-audio",
        model=MUSICGEN_REPO,
        device="cpu",           # change to 0 (or "cuda") if a GPU is available
        framework="pt",         # MusicGen uses PyTorch regardless of TF notebook context
    )

    print(f"Generating {MUSICGEN_DURATION}s of audio for prompt:\n  '{MUSICGEN_PROMPT}'")
    result = musicgen_pipe(
        MUSICGEN_PROMPT,
        forward_params={
            "do_sample": True,
            "max_new_tokens": int(MUSICGEN_DURATION * 50),
        },
    )

    audio = result["audio"].squeeze()
    sr    = result["sampling_rate"]

    # Normalise to int16 for WAV export
    audio_int16 = (audio / np.abs(audio).max() * 32767).astype(np.int16)
    scipy.io.wavfile.write(OUTPUT_WAV, sr, audio_int16)
    print(f"Saved to {OUTPUT_WAV}")

    ipd.display(ipd.Audio(audio, rate=sr))


### Option B — `sander-wood/tunesformer` (ABC seed → ABC notation)

TunesFormer is a GPT-style causal LM fine-tuned on thousands of Irish/Celtic folk
tunes in ABC notation.  You supply a short **seed** (title, metre, key, a bar or
two) and the model completes the tune character-by-character — the same mechanism
as the MIT from-scratch LSTM, but using a pretrained model.

The output is **ABC notation text** which you can:
- Copy into [https://abc.rectanglered.com](https://abc.rectanglered.com) to hear it
- Convert to MIDI with `music21` (shown below)
- Convert to audio with `timidity` or GarageBand

Run the cell below when `MUSIC_MODEL = "tunesformer"`.


In [ ]:
if MUSIC_MODEL == "tunesformer":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print(f"Loading {TUNESFORMER_REPO} ...")
    tf_tokenizer = AutoTokenizer.from_pretrained(TUNESFORMER_REPO)
    tf_lm_model  = AutoModelForCausalLM.from_pretrained(TUNESFORMER_REPO)
    tf_lm_model.eval()

    inputs = tf_tokenizer(TUNESFORMER_SEED, return_tensors="pt")

    print("Generating ABC notation ...")
    with torch.no_grad():
        output_ids = tf_lm_model.generate(
            inputs["input_ids"],
            max_new_tokens=TUNESFORMER_NEW_TOKENS,
            do_sample=True,
            temperature=TUNESFORMER_TEMPERATURE,
            top_k=TUNESFORMER_TOP_K,
            pad_token_id=tf_tokenizer.eos_token_id,
        )

    generated_abc = tf_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print("\n-- Generated ABC notation ------------------------------------------")
    print(generated_abc)
    print("--------------------------------------------------------------------")

    # Optional: convert to MIDI with music21
    try:
        from music21 import converter, midi
        score = converter.parse(generated_abc, format="abc")
        mf = midi.translate.music21ObjectToMidiFile(score)
        midi_path = "generated_tune.mid"
        mf.open(midi_path, "wb")
        mf.write()
        mf.close()
        print(f"MIDI saved to {midi_path}")
    except Exception as e:
        print(f"[music21 MIDI export skipped: {e}]")
        print("Tip: paste the ABC text above into https://abc.rectanglered.com to hear it.")
